# PBMC3k: an executable RustScenic RNA starter

**Audience:** Python users learning the expression-matrix input, GRN edge table,
and per-cell candidate gene-set scores. Basic NumPy/pandas familiarity helps.
Start with a **fresh Python 3.12 kernel**, internet access and enough memory for
a small single-cell example. Run the cells from top to bottom.

This notebook installs the **published `rustscenic==0.5.1` wheel**, downloads
the public 10x PBMC3k count archive, and runs inference. It does not build or
import the repository source. It uses all **2,700 cells**, a modest gene/TF
panel, two Rayon threads and fixed model settings. Runtime depends on the
machine; this is not a speed benchmark or a claim of testing on Google Colab.

By the end you will have:

1. Loaded genes-by-cells 10x data as a cells-by-genes AnnData matrix.
2. Run a GRN model and constructed top-target **candidate gene sets**.
3. Scored those sets with AUCell, inspected a plot, and exported results plus provenance.

**Method boundary:** these sets are **not motif-filtered, signed, or full
SCENIC regulons**. Edge importance and activity scores are exploratory model
outputs, not experimental evidence of regulation or cell-type validation.
See [method differences and validation limits](https://ekin-kahraman.github.io/rustscenic/validation/).
The [browser demo](https://ekin-kahraman.github.io/rustscenic/demo/endothelial/)
displays saved results; this notebook executes a different, PBMC3k workflow.

## 1. Install the public wheel and set the execution limits

This cell deliberately pins RustScenic to the public release rather than
installing from GitHub. Binary-only installation prevents an unexpected
local source build. Core dependencies already provide AnnData, NumPy,
SciPy and pandas; matplotlib is the only extra analysis dependency.
If RustScenic was imported earlier, restart the kernel before running all cells.

In [ ]:
%pip install --quiet --only-binary=:all: --index-url https://pypi.org/simple "rustscenic==0.5.1" "matplotlib>=3.9,<4"

In [ ]:
import os
import sys

if 'rustscenic' in sys.modules:
    raise RuntimeError('Restart the kernel, then run all cells to set the thread limits before import.')
THREADS = 2
os.environ['RAYON_NUM_THREADS'] = str(THREADS)
for name in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS'):
    os.environ[name] = '1'

import hashlib
import importlib.machinery
import importlib.metadata as metadata
import io
import json
import platform
import shutil
import tarfile
import time
import urllib.request
from datetime import datetime, timezone
from pathlib import Path

import anndata as ad
from IPython import get_ipython

get_ipython().run_line_magic('matplotlib', 'inline')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.sparse as sp
from IPython.display import display
from scipy.io import mmread

import rustscenic
import rustscenic._rustscenic as native
from rustscenic.aucell import score
from rustscenic.grn import infer

assert metadata.version('rustscenic') == '0.5.1'
assert rustscenic.__version__ == native.__version__ == '0.5.1'
package_path = Path(rustscenic.__file__).resolve()
native_path = Path(native.__file__).resolve()
assert package_path.is_relative_to(Path(sys.prefix).resolve()), 'Expected an installed package in this Python environment.'
assert any(str(native_path).endswith(suffix) for suffix in importlib.machinery.EXTENSION_SUFFIXES)
print(f'RustScenic {rustscenic.__version__}; Python {platform.python_version()}; Rayon threads={THREADS}')

## 2. Download and verify the public count archive

Source: **10x Genomics, “3k PBMCs from a Healthy Donor”, 26 May 2016,
Cell Ranger 1.1.0** ([dataset page](https://www.10xgenomics.com/datasets/3-k-pbm-cs-from-a-healthy-donor-1-standard-1-1-0)).
The provider makes the dataset available under
[CC BY 4.0](https://creativecommons.org/licenses/by/4.0/);
see its [citation guidance](https://www.10xgenomics.com/support/software/cell-ranger/latest/miscellaneous/cr-citations).
This example transposes the matrix, sums duplicate gene symbols, normalizes
counts, log-transforms and selects genes. These transformations are recorded below.

The compressed archive is about **7.6 MB**. Its SHA256 was calculated from
the downloaded official archive and is pinned here to detect changed/corrupt
input. We read only three named regular members; no archive paths are extracted.

In [ ]:
DATA_URL = 'https://cf.10xgenomics.com/samples/cell-exp/1.1.0/pbmc3k/pbmc3k_filtered_gene_bc_matrices.tar.gz'
DATA_SHA256 = '847d6ebd9a1ec9a768f2be7e40ca42cbfe75ebeb6d76a4c24167041699dc28b5'
WORK_DIR = Path.cwd() / 'rustscenic_pbmc3k'
DATA_DIR = WORK_DIR / 'data'
OUT_DIR = WORK_DIR / 'results'
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
archive = DATA_DIR / 'pbmc3k_filtered_gene_bc_matrices.tar.gz'

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

if not archive.exists():
    partial = archive.with_suffix('.download')
    request = urllib.request.Request(DATA_URL, headers={'User-Agent': 'RustScenic-PBMC3k-tutorial/0.5.1'})
    with urllib.request.urlopen(request, timeout=60) as response, partial.open('wb') as handle:
        shutil.copyfileobj(response, handle)
    if sha256_file(partial) != DATA_SHA256:
        partial.unlink()
        raise ValueError('Downloaded archive checksum differs from the documented dataset. Stop and investigate the source.')
    partial.replace(archive)
assert sha256_file(archive) == DATA_SHA256, 'Cached archive checksum mismatch; remove the cache and investigate before retrying.'

member_prefix = 'filtered_gene_bc_matrices/hg19/'
with tarfile.open(archive, mode='r:gz') as tar:
    def read_member(name):
        member = tar.getmember(member_prefix + name)
        assert member.isfile(), f'Expected a regular archive member: {name}'
        with tar.extractfile(member) as handle:
            return handle.read()

    raw_counts = sp.csr_matrix(mmread(io.BytesIO(read_member('matrix.mtx'))).T)
    genes = pd.read_csv(io.BytesIO(read_member('genes.tsv')), sep='\t', header=None,
                        names=['gene_id', 'gene_symbol'], dtype=str, keep_default_na=False)
    barcodes = pd.read_csv(io.BytesIO(read_member('barcodes.tsv')), header=None,
                           dtype=str, keep_default_na=False)[0].tolist()

assert raw_counts.shape == (2700, 32738)
assert raw_counts.shape == (len(barcodes), len(genes))
assert raw_counts.nnz == 2286884
assert pd.Index(barcodes).is_unique
assert genes['gene_symbol'].ne('').all()
assert np.isfinite(raw_counts.data).all() and (raw_counts.data >= 0).all()
print(f'Loaded {raw_counts.shape[0]:,} cells × {raw_counts.shape[1]:,} source genes; {raw_counts.nnz:,} nonzero counts')

## 3. Normalize all genes, then select a modest modelling panel

10x stores genes as rows. We transposed once, so **rows are cells and columns
are genes** throughout the analysis. Duplicate gene symbols are explicitly
summed **on raw counts**, with original gene IDs retained in the annotations.
Cell barcodes are never renamed or filtered.

Each cell is scaled to a library total of 10,000 **before gene subsetting**,
then `log1p` is applied. For a short starter, select the 600 most variable
log-normalized genes detected in at least three cells, plus available genes
from a fixed illustrative TF panel. This is simple variance ranking, **not**
Scanpy's highly-variable-gene procedure or a comprehensive TF catalogue.

In [ ]:
NORMALIZATION_TOTAL = 10_000.0
MIN_CELLS = 3
N_VARIABLE_GENES = 600
TF_PANEL = ['SPI1', 'CEBPB', 'CEBPD', 'IRF8', 'MAFB', 'PAX5', 'EBF1', 'BACH2',
            'TCF7', 'LEF1', 'ETS1', 'GATA3', 'RUNX3', 'TBX21', 'EOMES', 'KLF4',
            'ZEB2', 'STAT1', 'IRF1', 'NFKB1']

symbol_codes, symbols = pd.factorize(genes['gene_symbol'], sort=False)
symbol_sum = sp.csr_matrix((np.ones(len(genes), dtype=np.float32),
                           (np.arange(len(genes)), symbol_codes)),
                          shape=(len(genes), len(symbols)))
counts = (raw_counts.astype(np.float32) @ symbol_sum).tocsr()
counts.sum_duplicates()
counts.sort_indices()
library_totals = np.asarray(counts.sum(axis=1)).ravel()
assert (library_totals > 0).all(), 'This starter expects nonempty input cells.'
log_expression = counts.multiply((NORMALIZATION_TOTAL / library_totals)[:, None]).tocsr()
log_expression.data = np.log1p(log_expression.data).astype(np.float32)

detected_cells = np.asarray((counts > 0).sum(axis=0)).ravel()
means = np.asarray(log_expression.mean(axis=0)).ravel()
variances = np.maximum(np.asarray(log_expression.power(2).mean(axis=0)).ravel() - means**2, 0)
eligible = (detected_cells >= MIN_CELLS) & (variances > 0)
eligible_positions = np.flatnonzero(eligible)
ranked = eligible_positions[np.argsort(-variances[eligible_positions], kind='stable')]
symbol_index = pd.Index(symbols)
tfs = [tf for tf in TF_PANEL if tf in symbol_index and eligible[symbol_index.get_loc(tf)]]
missing_tfs = [tf for tf in TF_PANEL if tf not in tfs]
selected = np.union1d(ranked[:N_VARIABLE_GENES], symbol_index.get_indexer(tfs))
assert len(tfs) >= 2 and (selected >= 0).all()

gene_info = genes.groupby('gene_symbol', sort=False).agg(
    source_gene_ids=('gene_id', lambda values: ';'.join(values)),
    source_feature_count=('gene_id', 'size'),
).reindex(symbols)
gene_info.index = symbol_index
gene_info['detected_cells'] = detected_cells
gene_info['log_expression_variance'] = variances
adata = ad.AnnData(X=log_expression[:, selected].copy(),
                  obs=pd.DataFrame(index=pd.Index(barcodes, name='cell_barcode')),
                  var=gene_info.iloc[selected].copy())
adata.layers['counts'] = counts[:, selected].copy()
assert adata.shape[0] == 2700 and adata.obs_names.tolist() == barcodes
assert adata.var_names.is_unique and set(tfs).issubset(adata.var_names)
assert np.isfinite(adata.X.data).all()
print(f'Model input: {adata.n_obs:,} cells × {adata.n_vars:,} genes; {len(tfs)} TFs')
print(f'Duplicate source symbols merged: {len(genes) - len(symbols)}; unavailable/ineligible panel genes: {missing_tfs}')

## 4. Fit a small GRN model

The settings below are deliberately modest for learning. The seed, threads,
gene/TF order and model parameters are explicit. They do not establish
convergence, biological accuracy, a speedup, or equality to pySCENIC outputs.
The output contains candidate `TF → target` edges and unsigned importance.
A fixed top-50 output filter keeps the exported edge table small.

In [ ]:
GRN_SETTINGS = dict(n_estimators=100, learning_rate=0.05, max_features=0.5,
                    subsample=0.9, max_depth=2, early_stop_window=10,
                    early_stop_mode='arboreto', seed=777,
                    top_targets_per_tf=50, min_importance=None,
                    target_block_size=None, verbose=False)
started = time.perf_counter()
grn = infer(adata, tf_names=tfs, **GRN_SETTINGS)
grn_elapsed = time.perf_counter() - started
assert not grn.empty and {'TF', 'target', 'importance'}.issubset(grn.columns)
assert np.isfinite(grn['importance']).all() and (grn['importance'] > 0).all()
assert set(grn['TF']).issubset(tfs) and set(grn['target']).issubset(adata.var_names)
print(f'{len(grn):,} retained candidate edges (elapsed here: {grn_elapsed:.1f} s; not a benchmark)')
display(grn.sort_values('importance', ascending=False).head(5))

## 5. Build candidate gene sets and score them with AUCell

For each TF, keep up to 25 targets with the largest importance, breaking
importance ties by target name. There is **no motif enrichment/filtering or
activation/repression split**, so these are named candidate gene sets rather
than validated SCENIC regulons.

AUCell ranks genes **within this notebook's selected gene panel**. A `top_frac`
of 0.05 therefore refers to this restricted ranking universe, not all source
genes. Changing the selected genes, target-set sizes or cutoff changes the
scores; do not compare them as calibrated probabilities or validation metrics.

In [ ]:
TARGETS_PER_SET = 25
TOP_FRAC = 0.05
ordered_edges = grn.loc[grn['TF'] != grn['target']].sort_values(
    ['TF', 'importance', 'target'], ascending=[True, False, True], kind='stable')
candidate_sets = {
    f'{tf}_top{TARGETS_PER_SET}_candidates': group.head(TARGETS_PER_SET)['target'].tolist()
    for tf, group in ordered_edges.groupby('TF', sort=True)
}
assert candidate_sets and all(candidate_sets.values())
auc = score(adata, candidate_sets, top_frac=TOP_FRAC)
assert auc.shape == (adata.n_obs, len(candidate_sets))
assert auc.index.equals(adata.obs_names), 'Cell identity/order must be preserved.'
assert np.isfinite(auc.to_numpy()).all()
assert ((auc.to_numpy() >= 0) & (auc.to_numpy() <= 1)).all()
assert (auc.to_numpy() > 0).any(), 'Inspect the gene universe/cutoff if every score is zero.'
print(f'Scored {auc.shape[0]:,} cells × {auc.shape[1]} candidate sets; barcode order preserved')
display(auc.iloc[:5, :4])

## 6. Inspect score distributions

This plot is a software-output check: it shows the distributions of up to
six candidate-set scores across **all cells**. It does not assign cell types
or test an expected lineage. A high score means more of that set appears
near the top of the selected expression ranks, under this cutoff.

In [ ]:
plot_names = auc.columns[:6].tolist()
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.boxplot([auc[name].to_numpy() for name in plot_names], showfliers=False,
           tick_labels=[name.removesuffix('_candidates') for name in plot_names])
ax.set(xlabel='Candidate gene set', ylabel='AUCell score',
       title='Candidate-set scores across all 2,700 PBMC3k cells')
ax.tick_params(axis='x', labelrotation=30)
fig.tight_layout()
fig.savefig(OUT_DIR / 'candidate_score_distributions.png', dpi=140, bbox_inches='tight')
plt.show()

## 7. Export the inputs, results and run provenance

The result directory contains the selected log-normalized AnnData matrix
(with selected raw counts in `.layers['counts']`), exact gene/cell identifiers,
GRN edges, candidate sets, barcode-indexed scores and plot. `run_provenance.json`
records dataset/native-binary/output hashes, settings and dependency versions.
It describes this run, rather than promising bitwise agreement on every
platform or providing a complete environment lockfile.

In [ ]:
grn.to_csv(OUT_DIR / 'grn_edges.csv', index=False, float_format='%.17g')
auc.to_csv(OUT_DIR / 'candidate_auc.csv', index_label='cell_barcode', float_format='%.17g')
(OUT_DIR / 'candidate_gene_sets.json').write_text(json.dumps(candidate_sets, indent=2) + '\n')
adata.var.to_csv(OUT_DIR / 'selected_genes.tsv', sep='\t', index_label='gene_symbol')
adata.write_h5ad(OUT_DIR / 'selected_expression.h5ad', compression='gzip')

def json_native(value):
    if isinstance(value, np.generic):
        return value.item()
    raise TypeError(f'Unexpected metadata type: {type(value).__name__}')

result_files = sorted(path for path in OUT_DIR.iterdir()
                      if path.is_file() and path.name != 'run_provenance.json')
provenance = {
    'schema_version': 1,
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'rustscenic_version': metadata.version('rustscenic'),
    'python': platform.python_version(), 'platform': platform.platform(),
    'dependencies': {name: metadata.version(name) for name in
                     ['numpy', 'scipy', 'pandas', 'pyarrow', 'anndata', 'matplotlib']},
    'package_path': str(package_path), 'native_extension_path': str(native_path),
    'native_extension_sha256': sha256_file(native_path),
    'dataset': {'provider': '10x Genomics', 'title': '3k PBMCs from a Healthy Donor',
                'date': '2016-05-26', 'cell_ranger': '1.1.0',
                'landing_page': 'https://www.10xgenomics.com/datasets/3-k-pbm-cs-from-a-healthy-donor-1-standard-1-1-0',
                'url': DATA_URL, 'archive_sha256': DATA_SHA256,
                'archive_bytes': archive.stat().st_size, 'license': 'CC BY 4.0',
                'license_url': 'https://creativecommons.org/licenses/by/4.0/',
                'source_shape_cells_genes': list(raw_counts.shape)},
    'preprocessing': {'orientation': 'cells_by_genes', 'all_source_cells_retained': True,
                      'duplicate_symbols': 'sum raw counts before normalization; retain source gene IDs',
                      'normalization': 'library_total_then_log1p_before_gene_selection',
                      'normalization_total': NORMALIZATION_TOTAL, 'minimum_detected_cells': MIN_CELLS,
                      'selection': 'top variance in log-normalized expression plus eligible TF panel',
                      'variable_gene_limit': N_VARIABLE_GENES,
                      'selected_shape': list(adata.shape),
                      'tf_panel_requested': TF_PANEL, 'tf_panel_used': tfs,
                      'tf_panel_missing_or_ineligible': missing_tfs},
    'threads': {name: os.environ[name] for name in
                ['RAYON_NUM_THREADS', 'OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS']},
    'grn_settings': GRN_SETTINGS, 'grn_fit': grn.attrs.get('grn_fit', {}),
    'grn_elapsed_seconds_this_run': grn_elapsed,
    'candidate_sets': {'construction': 'top unsigned importance targets per TF; target-name tie break',
                       'maximum_targets_per_set': TARGETS_PER_SET,
                       'motif_filtered': False, 'signed': False},
    'aucell': {'top_frac': TOP_FRAC, 'ranking_universe_genes': adata.n_vars,
               'coverage': auc.attrs.get('regulon_coverage', {}),
               'shape': list(auc.shape), 'cell_order_preserved': auc.index.equals(adata.obs_names)},
    'output_sha256': {path.name: sha256_file(path) for path in result_files},
}
provenance_path = OUT_DIR / 'run_provenance.json'
provenance_path.write_text(json.dumps(provenance, indent=2, allow_nan=False, default=json_native) + '\n')
reloaded = pd.read_csv(OUT_DIR / 'candidate_auc.csv', index_col=0, dtype={'cell_barcode': str})
assert reloaded.index.tolist() == barcodes
assert reloaded.shape == auc.shape
print(f'Exported {len(result_files) + 1} files to {OUT_DIR}')
display(pd.DataFrame({'file': [path.name for path in result_files] + [provenance_path.name]}))

## Optional exercise: change one candidate-set definition

Choose a TF from `tfs` and reduce the number of retained targets to ten.
How does the candidate-set size affect its scores? The runnable answer
scaffold below scores that one new set without refitting the GRN or changing
the main exported results. Differences reflect a changed set definition,
not evidence that one model is biologically better.

In [ ]:
EXERCISE_TF = ordered_edges['TF'].iloc[0]  # Try another TF with retained edges.
EXERCISE_TARGETS = 10
exercise_genes = ordered_edges.loc[ordered_edges['TF'] == EXERCISE_TF, 'target'].head(EXERCISE_TARGETS).tolist()
assert exercise_genes, 'Choose a TF that has retained candidate edges.'
exercise_name = f'{EXERCISE_TF}_top{EXERCISE_TARGETS}_exercise'
exercise_auc = score(adata, {exercise_name: exercise_genes}, top_frac=TOP_FRAC)
assert exercise_auc.index.equals(adata.obs_names)
display(exercise_auc.describe().loc[['mean', 'min', 'max']])

## What to check before extending this example

- **Keep the axes and identifiers aligned.** Genes are columns; candidate
  names must match their gene symbols, and score rows must match cell barcodes.
- **Normalize before subsetting.** Scaling only the selected panel changes
  the library denominator. This notebook retains the full-library denominator.
- **Interpret the rank universe.** AUCell here uses the selected panel.
  Changing that panel changes the meaning and cutoff of its scores.
- **Use a complete method for scientific conclusions.** Larger TF/gene panels,
  parameter/convergence checks, motif-supported regulatory definitions and
  appropriate independent evaluation are separate work, not demonstrated here.

[GRN API](https://ekin-kahraman.github.io/rustscenic/reference/grn/) ·
[AUCell API](https://ekin-kahraman.github.io/rustscenic/reference/aucell/) ·
[Validation limits](https://ekin-kahraman.github.io/rustscenic/validation/)

This starter is maintained with RustScenic. AI tools assisted notebook
implementation and validation; the recorded execution environment and
outcomes should be consulted for the actual tested scope.